# [Introduction | Online Z3 Guide](https://microsoft.github.io/z3guide/programming/Z3%20Python/Introduction)

- **学習日**：2026-11-08
- **対象範囲**：Kinematic Equations ～ ページ末尾
  - *Bit Tricksまでしかやってない*
- **内容**：Z3Pyを用いた具体的な問題解決・応用例

## Kinematic Equations

- displacement (d), 
- time (t), 
- acceleration (a), 
- initial velocity (v_i) and 
- final velocity (v_f).

### Problem 1

$$
\begin{aligned}
d &= v_i t+\frac12at^2\\
v_f &= v_i+at
\end{aligned}
$$

条件
$$
v_i=30,\qquad v_f=0,\qquad a=-8
$$

In [3]:
from z3 import ArithRef, BoolRef, Reals, solve

# Variables
d: ArithRef
a: ArithRef
t: ArithRef
v_i: ArithRef
v_f: ArithRef

d, a, t, v_i, v_f = Reals("d a t v_i v_f")

# Kinematic equations
equations: list[BoolRef] = [  # pyright: ignore[reportAssignmentType]
    d == v_i * t + (a * t**2) / 2,  # pyright: ignore[reportOperatorIssue]
    v_f == v_i + a * t,  # pyright: ignore[reportOperatorIssue]
]

print("Kinematic equations:")
print(equations)
print()

# Given conditions
problem: list[BoolRef] = [  # pyright: ignore[reportAssignmentType]
    v_i == 30,
    v_f == 0,
    a == -8,
]

print("Problem:")
print(problem)
print()

print("Solution:")
solve(equations + problem)

Kinematic equations:
[d == v_i*t + (a*t**2)/2, v_f == v_i + a*t]

Problem:
[v_i == 30, v_f == 0, a == -8]

Solution:
[v_i = 30, v_f = 0, d = 225/4, t = 15/4, a = -8]


#### 実数算術を背景Theoryとする多ソート一階論理として見ると

$$
\Gamma=
\left\{
\begin{aligned}
d &= v_i t+\frac12at^2\\
v_f &= v_i+at\\
v_i &=30\\
v_f &=0\\
a &=-8
\end{aligned}
\right\}
$$

これらをすべて満たす解釈をZ3に探索させている。

#### FOLの構成要素との対応

| 概念 | 今回の例 | 正確な位置づけ |
| --- | --- | --- |
| 個体（Individual） | $30,\ 3.75,\ 56.25$ など | 領域 $\mathbb R$ の要素。実数値そのもの |
| 変数（Variable） | 明示的には登場しない | FOLの量化変数。今回のZ3コードには量化子がない |
| 定数記号（Constant Symbol） | $d,a,t,v_i,v_f$ | Z3の `Reals()` で生成した、解釈が未確定の0引数記号 |
| 数値定数（Numeral） | $0,2,30,-8$ | 実数算術Theoryによって意味が固定された数値リテラル |
| 関数記号（Function Symbol） | $+,-,\times,\div$ | 実数上の演算。背景Theoryによって解釈済み |
| 述語記号（Predicate Symbol） | $=$ | 組込みの等号。二つの項の等しさを表す |
| 論理式（Formula） | $v_f=v_i+at$ など | 等式原子式。これらを連言して制約集合を構成 |

### Problem 2

In [5]:
from z3 import BoolRef, RealVal, set_option, solve

# Given v_i, t and a, find d
problem: list[BoolRef] = [  # pyright: ignore[reportAssignmentType]
    v_i == 0,
    t == RealVal("4.10"),
    a == 6,
]

# 有理数表示
set_option(rational_to_decimal=False)
solve(*(equations + problem))

# 小数表示
set_option(rational_to_decimal=True)
solve(*(equations + problem))

[a = 6, v__i = 0, d = 5043/100, v__f = 123/5, t = 41/10]
[a = 6, v__i = 0, d = 50.43, v__f = 24.6, t = 4.1]


## Bit Tricks

C言語などで使われる高速なビット演算が正しいことを、Z3で証明する

### Power of two

32ビットのビット列 $x$ について、次の式で符号なしの値が2の累乗か判定できる。

$$
\boxed{x\neq0\ \land\ (x\ \&\ (x-1))=0}
$$

2の累乗は1のビットがちょうど1個あり、1を引くとそのビットが消える。$x=0$ も後半の条件を満たすため、別途除外する。最上位ビットだけが1の場合も、符号なしの値としては2の累乗に含める。

In [7]:
from z3 import And, BitVec, BitVecRef, BoolRef, Or, prove

x: BitVecRef = BitVec("x", 32)

powers: list[int] = [2**i for i in range(32)]

# 高速なビット演算による判定
fast: BoolRef = And(  # pyright: ignore[reportAssignmentType]
    x != 0,
    (x & (x - 1)) == 0,  # pyright: ignore[reportOperatorIssue]
)
print("fast: \n", fast, "\n")

# 2の累乗をすべて列挙した判定
slow: BoolRef = Or([x == p for p in powers])  # pyright: ignore[reportAssignmentType]
print("slow: \n", slow, "\n")

# 両者が常に一致することを証明
prove(fast == slow)
print()

print("Trying to prove buggy version...")

# x != 0 を忘れたバグ
buggy: BoolRef = (x & (x - 1)) == 0  # pyright: ignore[reportOperatorIssue, reportAssignmentType]

prove(buggy == slow)

fast: 
 And(x != 0, x & x - 1 == 0) 

slow: 
 Or(x == 1,
   x == 2,
   x == 4,
   x == 8,
   x == 16,
   x == 32,
   x == 64,
   x == 128,
   x == 256,
   x == 512,
   x == 1024,
   x == 2048,
   x == 4096,
   x == 8192,
   x == 16384,
   x == 32768,
   x == 65536,
   x == 131072,
   x == 262144,
   x == 524288,
   x == 1048576,
   x == 2097152,
   x == 4194304,
   x == 8388608,
   x == 16777216,
   x == 33554432,
   x == 67108864,
   x == 134217728,
   x == 268435456,
   x == 536870912,
   x == 1073741824,
   x == 2147483648) 

proved

Trying to prove buggy version...
counterexample
[x = 0]


### Opposite signs

符号付き整数の2の補数表現では、最上位ビットが符号を決める。

ビットごとの排他的論理和（XOR）を取れば、最上位ビットが異なるかを判定できる。
$$
\boxed{(x\oplus y)<0}
$$

ここで `^` はビット単位のXORで、$<0$ は符号付き比較を表す。ゼロは非負として扱う。

In [ ]:
from z3 import And, BitVec, BitVecRef, BoolRef, Or, prove

x: BitVecRef = BitVec("x", 32)
y: BitVecRef = BitVec("y", 32)

# XORで符号の違いを判定
trick: BoolRef = (x ^ y) < 0  # pyright: ignore[reportOperatorIssue]

# 普通の方法
opposite: BoolRef = Or(  # pyright: ignore[reportAssignmentType]
    And(x < 0, y >= 0),
    And(x >= 0, y < 0),
)

prove(trick == opposite)